
## Batch Ingestion
Every time you run the cell, it will read all files from the folder and load them into the table

In [0]:
%sql
DROP TABLE IF EXISTS perfume;
    
CREATE TABLE perfume 
SELECT *
FROM read_files(
  -- get the parquet data from a shared delta directory
  '/Volumes/aidc_inc_luxury_perfume_prices_full/aidcdatasetshare/full_free_2025_luxury_perfume_prices',
  format => 'parquet'
);
    
SELECT * FROM perfume LIMIT 10

In [0]:
%sql
DESCRIBE TABLE EXTENDED perfume 

In [0]:
# Doing the same thing in Python
# load table from folder
df = spark.read.format('parquet').load('/Volumes/aidc_inc_luxury_perfume_prices_full/aidcdatasetshare/full_free_2025_luxury_perfume_prices')

# Write table
df.write.mode('overwrite').saveAsTable('workspace.default.perfume')

# view the created table
perfume = spark.table('workspace.default.perfume')
perfume.display()


### Incremental Load (COPY INTO)

In [0]:
%sql
DROP TABLE IF EXISTS perfume;

CREATE TABLE perfume (
    brand STRING -- incomplete schema, you can also skip creating it
);

COPY INTO perfume
    FROM '/Volumes/aidc_inc_luxury_perfume_prices_full/aidcdatasetshare/full_free_2025_luxury_perfume_prices'
    FILEFORMAT = PARQUET
    COPY_OPTIONS ('mergeSchema' = 'true'); -- Merge the existing schema with the actual table schema
    
SELECT * FROM perfume LIMIT 10


In [0]:
%sql
-- Run it again to see no row affected due to incremental load
COPY INTO perfume
    FROM '/Volumes/aidc_inc_luxury_perfume_prices_full/aidcdatasetshare/full_free_2025_luxury_perfume_prices'
    FILEFORMAT = PARQUET
    COPY_OPTIONS ('mergeSchema' = 'true'); 

## Incremental Ingestion with Auto Loader 
CREATE STREAMING TABLE is recommended comparing to COPY INTO for ingestion from cloud storage

`Note: Creating streaming tables with Databrick SQL requires a SQL warehouse.`


In [0]:
%sql
-- Create streaming table
CREATE OR REFRESH STREAMING TABLE perfume_autoloader
SCHEDULE EVERY 1 WEEK -- scheduling the refresh is optional
AS
SELECT * 
FROM STREAM read_files( -- stream is the keyword as it indicates that this is a incremental load, not pure batch load
  '/Volumes/aidc_inc_luxury_perfume_prices_full/aidcdatasetshare/full_free_2025_luxury_perfume_prices',
  format => 'parquet'
);

-- This will take some time to run as it is creating a declarative pipeline for the streaming ingestion

In [0]:
%sql
DESCRIBE TABLE EXTENDED perfume_autoloader

In [0]:
%sql
DESCRIBE HISTORY perfume_autoloader

You can **manully refresh the streaming table** when new file arrives (it will only incrementally load the new file arrived in the folder)

In [0]:
%sql
REFRESH STREAMING TABLE perfume_autoloader

In [0]:
%sql
DESCRIBE HISTORY perfume_autoloader

In [0]:
%sql
-- Clean the table
DROP TABLE IF EXISTS perfume_autoloader

### Incremental batch with Auto Loader in Python

In [0]:
# Create a new volume as the check point
spark.sql(f"CREATE VOLUME IF NOT EXISTS check_point_volume")

# Use the existing volume in the workspace as the checkpoint
checkpoint = f'/Volumes/workspace/default/check_point_volume'

# Use Auto Loader for Incremental Batch
(spark.readStream.format("cloudFiles") 
  .option("cloudFiles.format", "csv") 
  .option("header", "true") 
  .option("inferSchema", "true") 
  .option("cloudFiles.schemaLocation", f"{checkpoint}") 
  .load(f"/Volumes/workspace/default/test_volume/test_directory/")  # read the data from the specified volume
.writeStream 
  .option("checkpointLocation", f"{checkpoint}") 
  .trigger(once=True) 
  .toTable(f"workspace.default.perfume_auto_loader") # write the read data into a specified table
)

In [0]:
%sql
DESCRIBE HISTORY perfume_auto_loader

In [0]:
%sql
DROP TABLE IF EXISTS perfume_auto_loader; 
DROP VOlUME IF EXISTS check_point_volume


## Adding Metadata Columns on Ingest to Bronze Layer
### You can add extra columns when ingesting to Bronze Layer
1. Include input file name: `_metadata.file_name ` 
2. Include last modification timestamp: `_metadata.file_modification_time`
3. Include file ingestion time: `current_timestamp()`

In [0]:
%sql
DROP TABLE IF EXISTS perfume_bronze;

-- Create the bronze table from the data sources with metadata
CREATE TABLE perfume_bronze AS
SELECT *,
  _metadata.file_modification_time AS file_mod_time,
  _metadata.file_name AS source_file,
  current_timestamp() AS ingestion_time
FROM read_files(
  '/Volumes/aidc_inc_luxury_perfume_prices_full/aidcdatasetshare/full_free_2025_luxury_perfume_prices',
  format => 'parquet'
);

SELECT * FROM perfume_bronze

#### Explore the bronze table
Count how many rows are from each file

In [0]:
%sql
SELECT source_file, 
  COUNT(*) AS file_count
FROM perfume_bronze
GROUP BY source_file
ORDER BY source_file

## Rescue Data Column


In [0]:
%sql
-- Define the rescue data column 
SELECT *
FROM read_files(
  '/Volumes/workspace/default/test_volume/test_directory/',
  format => 'csv',
  delimiter => ',',
  header => true,
  schema => '''user_query STRING, 
            search_result STRING''',
  rescuedDataColumn => '_rescued_data' -- need to manually create the rescue column if defining the schema manually
)

If your source data is missing column header like this, where the first column "_c0" header is missing, you can use below method (you can apply those cleaning/fixing methods when going from Bronze to Silver)

![image_1774836504285.png](./image_1774836504285.png "image_1774836504285.png")

In [0]:
%sql
SELECT
  cast(_rescued_data: _c0 AS INT) AS order_id, -- cast all the _c0 values into INT
  *
  FROM read_files(
    '/Volumes/workspace/default/test_volume/test_directory/',
    format => 'csv',
    delimiter => ',',
    header => true,
    schema => '''user_query STRING, 
              search_result STRING''',
    rescuedDataColumn => '_rescued_data' -- need to manually create the rescue column if defining the schema manually
  )

## Ingesting Semi-Structured Data: JSON

In [0]:
%sql
-- Create the bronze table from the JSON file directly
DROP TABLE IF EXISTS business_daily_bronze;

CREATE TABLE IF NOT EXISTS business_daily_bronze
SELECT *,
  _metadata.file_modification_time AS file_mod_time,
  _metadata.file_name AS source_file FROM read_files(
  '/Volumes/databricks_simulated_retail_customer_data/v02/business_daily_events',
  format => 'json'
);

SELECT * FROM business_daily_bronze

In [0]:
%sql
select cast(timestamp AS timestamp) from business_daily_bronze 

Use below methods if a string column actually holds json values

In [0]:
%sql
DROP TABLE IF EXISTS customer_json_row;

-- Create the table with each row being a JSON file
CREATE TABLE IF NOT EXISTS customer_json_row
SELECT *,
  _metadata.file_modification_time AS file_mod_time,
  _metadata.file_name AS source_file FROM read_files(
  '/Volumes/databricks_simulated_retail_customer_data/v02/customer_changes_daily',
  format => 'text');

SELECT * FROM customer_json_row

#### Method 1: Directly query the json value 
Cons: 
- Not performant
- No Schema (data integrity issue)
- Complex to query


In [0]:
%sql
-- Method 1: Directly query it by using column_name:key
SELECT value:customer_id,
       to_date(value:signup_date) AS signup_date
FROM customer_json_row
    

#### Method 2: Flatten the JSON via STRUCT 
CONs:
- Schema Enforced (issues if JSON evolves)
- Low flexibility

2 Steps:
1. Get the STRUCT (schema) of the JSON string
2. Apply the STRUCT tro the JSON string

In [0]:
%sql
-- Get the STRUCT or Schema by providing one example as a string
select schema_of_json('{"customer_id": "CUST_00164", "first_name": "Emma", "last_name": "Jenkins", "email": "emma.jenkins00164@example.com", "city": "London", "signup_date": "2025-11-01", "source_subsidiary": "northstar_outfitters", "loyalty_tier": "gold", "operation": "new", "timestamp": "2025-11-01T14:03:53Z"}') AS json_schema

In [0]:
%sql
-- Apply the schema to the json column (by copying the inferred schema above as a string) and update the table
CREATE OR REPLACE TABLE business_daily_bronze
AS
SELECT * except(value), -- need to exclude the json string column as it will be appended by the from_json()
    from_json(value, 'STRUCT<city: STRING, customer_id: STRING, email: STRING, first_name: STRING, last_name: STRING, loyalty_tier: STRING, operation: STRING, signup_date: STRING, source_subsidiary: STRING, timestamp: STRING>') AS value -- from_json() needs both the json string column and the inferred schema as parameters 
FROM customer_json_row;

SELECT * FROM business_daily_bronze


In [0]:
%sql
-- now we can query the json STRUCT column easily by using '.', not ':'
SELECT value.customer_id,
       to_date(value.signup_date) AS signup_date
FROM business_daily_bronze

Note: If you have an array being the value in the json string as a key-value pair, then you can use `explode()` to flatten that array (replicate each row for every array item).

Ex: SELECT explode(value.items) AS item_in_array

#### Use the VARIANT column (parse_json())
Very performant and flexible

In [0]:
%sql
CREATE OR REPLACE TABLE business_daily_bronze AS
SELECT * EXCEPT(value), 
  parse_json(value) AS variant_value -- covert the json string column into a variant data type column
FROM customer_json_row;

SELECT * FROM business_daily_bronze

In [0]:
%sql
SELECT variant_value:customer_id :: STRING, -- still need to use ':' to access the json value, and need to use '::' to cast the type from VARIANT into STRING
       variant_value:signup_date :: DATE AS signup_date -- Use '::' to directly cast the column to a specific type
FROM business_daily_bronze

In [0]:
%sql
DROP TABLE IF EXISTS customer_json_row;
DROP TABLE IF EXISTS business_daily_bronze;

## Data Ingestion with MERGE INTO

In [0]:
%sql
-- create the target table
DROP TABLE IF EXISTS business_target;

CREATE TABLE business_target AS
select * from 
read_files("/Volumes/databricks_simulated_retail_customer_data/v02/business_daily_events/retail_business_events_2025-11-01.json",
format => 'json');

SELECT * FROM business_target


In [0]:
%sql
-- Create the source table
DROP TABLE IF EXISTS business_source;

CREATE TABLE business_source AS
select * from 
read_files("/Volumes/databricks_simulated_retail_customer_data/v02/business_daily_events/retail_business_events_2025-11-02.json",
format => 'json');

SELECT * FROM business_source

In [0]:
%sql
-- Merge the 2 tables 
MERGE INTO business_target AS target
USING business_source AS source
ON target.batch_id = source.batch_id
WHEN MATCHED THEN 
  UPDATE SET *
WHEN NOT MATCHED THEN 
  INSERT *
    

In [0]:
%sql
-- The merge will auto-save the update to the target table 
SELECT * FROM business_target

In [0]:
%sql
DESCRIBE HISTORY business_target

In [0]:
%sql
-- Check the original target table
SELECT * FROM business_target VERSION AS OF 0

Note that if the source table have new columns (need to modify target table schema), then `MERGE INTO` will fail as it enforces the schema. 

To enable schema evolution if the new columns from the source tables are desired, use 
- `MERGE WITH SCHEMA EVOLUTION INTO`

In [0]:
%sql
-- Merge the 2 tables 
MERGE WITH SCHEMA EVOLUTION INTO business_target AS target
USING (
  SELECT * FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY batch_id ORDER BY timestamp DESC) as rn
    FROM business_source
  )
  WHERE rn = 1
) AS source
ON target.batch_id = source.batch_id
WHEN MATCHED THEN 
  UPDATE SET *
WHEN NOT MATCHED THEN 
  INSERT *;
